# 04 - Fine-Tuning Training (Google Colab)
Load processed dataset, configure Trainer with LoRA adapters, run fine-tuning loop, and save trained weights.

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import json
from pathlib import Path

import yaml
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

In [ ]:
# Load configurations
with open("configs/training_config.yaml", "r") as f:
    train_cfg = yaml.safe_load(f)

with open("configs/lora_config.yaml", "r") as f:
    lora_cfg = yaml.safe_load(f)

print("Training hyperparameters:")
for key in ["per_device_train_batch_size", "gradient_accumulation_steps",
            "learning_rate", "num_train_epochs", "warmup_steps"]:
    print(f"  {key}: {train_cfg[key]}")

effective_batch = train_cfg["per_device_train_batch_size"] * train_cfg["gradient_accumulation_steps"]
print(f"  effective_batch_size: {effective_batch}")

In [ ]:
# Step 1: Load processed dataset
print("Loading processed datasets...")

with open(train_cfg["train_dataset"], "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(train_cfg["validation_dataset"], "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Verify samples have 'text' field
assert "text" in train_data[0], "Run notebook 02 first to prepare dataset with formatted prompts"
print(f"\nSample text length: {len(train_data[0]['text'])} chars")

In [ ]:
# Step 2: Load model and tokenizer
model_name = train_cfg["model_name"]
use_4bit = train_cfg.get("use_4bit", True)

# Quantization config
bnb_config = None
if use_4bit and torch.cuda.is_available():
    compute_dtype = getattr(torch, train_cfg.get("bnb_4bit_compute_dtype", "float16"))
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=train_cfg.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=train_cfg.get("use_double_quant", True),
    )
    print("4-bit quantization enabled")

# Tokenizer
print(f"\nLoading tokenizer: {model_name}")
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
except Exception:
    model_name = train_cfg["fallback_model"]
    print(f"Fallback: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Model
print(f"Loading model: {model_name}")
model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
    "device_map": "auto",
}
if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
print(f"Model loaded: {type(model).__name__}")

In [ ]:
# Step 3: Apply LoRA adapters
if train_cfg.get("gradient_checkpointing", True):
    model.gradient_checkpointing_enable()

if bnb_config:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Step 4: Tokenize datasets
max_seq_length = train_cfg.get("max_seq_length", 2048)

def tokenize_data(data, tokenizer, max_length):
    texts = [sample["text"] for sample in data]

    def tokenize_fn(examples):
        tokenized = tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    dataset = Dataset.from_dict({"text": texts})
    return dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

print(f"Tokenizing with max_seq_length={max_seq_length}...")
train_dataset = tokenize_data(train_data, tokenizer, max_seq_length)
val_dataset = tokenize_data(val_data, tokenizer, max_seq_length)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")
print(f"Features: {train_dataset.features}")

In [ ]:
# Step 5: Configure training arguments
output_dir = train_cfg["output_dir"]

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=train_cfg.get("num_train_epochs", 3),
    per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 2),
    per_device_eval_batch_size=train_cfg.get("per_device_eval_batch_size", 2),
    gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 8),
    learning_rate=train_cfg.get("learning_rate", 2e-4),
    warmup_steps=train_cfg.get("warmup_steps", 100),
    logging_steps=train_cfg.get("logging_steps", 10),
    eval_steps=train_cfg.get("eval_steps", 50),
    save_steps=train_cfg.get("save_steps", 100),
    optim=train_cfg.get("optim", "adamw_torch"),
    lr_scheduler_type=train_cfg.get("lr_scheduler_type", "linear"),
    weight_decay=train_cfg.get("weight_decay", 0.01),
    fp16=train_cfg.get("fp16", True),
    bf16=train_cfg.get("bf16", False),
    save_total_limit=train_cfg.get("save_total_limit", 3),
    load_best_model_at_end=train_cfg.get("load_best_model_at_end", True),
    eval_strategy=train_cfg.get("evaluation_strategy", "steps"),
    save_strategy=train_cfg.get("save_strategy", "steps"),
    logging_dir=train_cfg.get("logging_dir", f"{output_dir}/logs"),
    report_to=train_cfg.get("report_to", "none"),
    seed=train_cfg.get("seed", 42),
    gradient_checkpointing=train_cfg.get("gradient_checkpointing", True),
)

print(f"Output directory: {output_dir}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")

In [ ]:
# Step 6: Create Trainer
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Trainer configured successfully")
print(f"Training steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")

In [ ]:
# Step 7: Run fine-tuning
print("Starting fine-tuning...")
print("=" * 50)

train_result = trainer.train()

print("\n" + "=" * 50)
print("Training complete!")
print(f"\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Step 8: Save LoRA adapter weights
print(f"Saving LoRA adapter to: {output_dir}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("\nSaved files:")
for f in sorted(Path(output_dir).glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name}: {size:.1f} KB")

In [ ]:
# Step 9: Final evaluation
print("Running final evaluation...")
eval_results = trainer.evaluate()

print(f"\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value}")

print(f"\nFine-tuning complete!")
print(f"LoRA adapter saved to: {output_dir}")